In [1]:
import numpy as np

np.random.seed(42)

In [2]:
class LinearRegression:
    def __init__(self, learning_rate=0.01):
        self.w = 0.0
        self.b = 0.0
        self.lr = learning_rate
        self.cost_history = []

    def predict(self, X):
        # X shape: (n,)
        return self.w * X + self.b

    def compute_cost(self, X, y):
        predictions = self.predict(X)
        return np.mean((predictions - y) ** 2)

    def compute_gradients(self, X, y):
        predictions = self.predict(X)
        n = len(y)
        errors = predictions - y
        dw = (2 / n) * np.sum(errors * X)
        db = (2 / n) * np.sum(errors)
        return dw, db

    def fit(self, X, y, epochs=1000, print_every=200):
        for epoch in range(epochs):
            dw, db = self.compute_gradients(X, y)
            self.w -= self.lr * dw
            self.b -= self.lr * db
            cost = self.compute_cost(X, y)
            self.cost_history.append(cost)
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | w: {self.w:.4f} | b: {self.b:.4f}")
        return self

    def r_squared(self, X, y):
        predictions = self.predict(X)
        ss_res = np.sum((y - predictions) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

In [3]:
class MultipleLinearRegression:
    def __init__(self, n_features, learning_rate=0.01):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate
        self.cost_history = []

    def predict(self, X):
        # X shape: (n, p), weights shape: (p,)
        return X @ self.weights + self.bias

    def compute_cost(self, X, y):
        predictions = self.predict(X)
        return np.mean((predictions - y) ** 2)

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = predictions - y                       # shape (n,)
            # Vectorized gradient: X.T @ errors → shape (p,)
            grad_w = (2 / n) * (X.T @ errors)
            grad_b = (2 / n) * np.sum(errors)
            self.weights -= self.lr * grad_w
            self.bias    -= self.lr * grad_b
            cost = self.compute_cost(X, y)
            self.cost_history.append(cost)
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f}")
        return self

    def r_squared(self, X, y):
        predictions = self.predict(X)
        ss_res = np.sum((y - predictions) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

In [4]:
class PolynomialRegression:
    def __init__(self, degree, learning_rate=0.01):
        self.degree = degree
        self.weights = np.zeros(degree)
        self.bias = 0.0
        self.lr = learning_rate

    def make_features(self, X):
        # X shape: (n,), output shape: (n, degree)
        # Column d holds X**(d+1)
        X = np.asarray(X)
        return np.column_stack([X ** (d + 1) for d in range(self.degree)])

    def predict(self, X):
        features = self.make_features(X)
        return features @ self.weights + self.bias

    def fit(self, X, y, epochs=1000, print_every=200):
        features = self.make_features(X)          # (n, degree)
        n = len(y)
        for epoch in range(epochs):
            predictions = features @ self.weights + self.bias
            errors = predictions - y
            grad_w = (2 / n) * (features.T @ errors)
            grad_b = (2 / n) * np.sum(errors)
            self.weights -= self.lr * grad_w
            self.bias    -= self.lr * grad_b
            if epoch % print_every == 0:
                cost = np.mean(errors ** 2)
                print(f"  Epoch {epoch:4d} | Cost: {cost:.6f}")
        return self

    def r_squared(self, X, y):
        predictions = self.predict(X)
        ss_res = np.sum((y - predictions) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

In [5]:
class RidgeRegression:
    def __init__(self, n_features, learning_rate=0.01, alpha=1.0):
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.lr = learning_rate
        self.alpha = alpha

    def predict(self, X):
        return X @ self.weights + self.bias

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = predictions - y
            mse = np.mean(errors ** 2)
            reg_term = self.alpha * np.sum(self.weights ** 2)
            cost = mse + reg_term

            # Gradient: MSE gradient + ridge gradient
            grad_w = (2 / n) * (X.T @ errors) + 2 * self.alpha * self.weights
            grad_b = (2 / n) * np.sum(errors)                 # no penalty on bias

            self.weights -= self.lr * grad_w
            self.bias    -= self.lr * grad_b

            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | L2 penalty: {reg_term:.4f}")
        return self

In [6]:
import numpy as np
np.random.seed(42)

# ============================================================
# 1. SIMPLE LINEAR REGRESSION DEMO
# ============================================================
print("=" * 60)
print("1. SIMPLE LINEAR REGRESSION")
print("=" * 60)

TRUE_W, TRUE_B = 3.5, 2.0
X_lin = np.random.uniform(0, 10, 200)
y_lin = TRUE_W * X_lin + TRUE_B + np.random.normal(0, 1.5, 200)

model_lin = LinearRegression(learning_rate=0.005)
model_lin.fit(X_lin, y_lin, epochs=2000, print_every=500)
print(f"\nLearned: y = {model_lin.w:.4f}x + {model_lin.b:.4f}")
print(f"True:    y = {TRUE_W}x + {TRUE_B}")
print(f"R-squared: {model_lin.r_squared(X_lin, y_lin):.4f}")

# ============================================================
# 2. MULTIPLE LINEAR REGRESSION DEMO
# ============================================================
print("\n" + "=" * 60)
print("2. MULTIPLE LINEAR REGRESSION (3 features)")
print("=" * 60)

N = 100
size     = np.random.uniform(500, 3000, N)
bedrooms = np.random.randint(1, 6, N)
age      = np.random.uniform(0, 50, N)
price    = (50 * size + 10000 * bedrooms - 1000 * age + 50000
            + np.random.normal(0, 20000, N))

X_multi = np.column_stack([size, bedrooms, age])
y_multi = price

# Standardize features
def standardize(X):
    means = X.mean(axis=0)
    stds  = X.std(axis=0)
    stds[stds == 0] = 1.0
    return (X - means) / stds, means, stds

X_scaled, x_means, x_stds = standardize(X_multi)
y_mean_val = y_multi.mean()
y_std_val  = y_multi.std()
y_scaled   = (y_multi - y_mean_val) / y_std_val

multi_model = MultipleLinearRegression(n_features=3, learning_rate=0.01)
multi_model.fit(X_scaled, y_scaled, epochs=1000, print_every=200)

print(f"\nWeights (standardized): {np.round(multi_model.weights, 4)}")
print(f"Bias (standardized):    {multi_model.bias:.4f}")
print(f"R-squared:              {multi_model.r_squared(X_scaled, y_scaled):.4f}")

# ============================================================
# 3. POLYNOMIAL REGRESSION DEMO
# ============================================================
print("\n" + "=" * 60)
print("3. POLYNOMIAL REGRESSION (degree 2 vs 5)")
print("=" * 60)

X_poly_raw = np.arange(0, 50) / 10.0
y_poly_raw = (0.5 * X_poly_raw ** 2 - 2 * X_poly_raw + 3
              + np.random.normal(0, 1.0, len(X_poly_raw)))

x_max     = np.max(np.abs(X_poly_raw))
X_poly_n  = X_poly_raw / x_max
y_poly_mu = y_poly_raw.mean()
y_poly_sd = y_poly_raw.std()
y_poly_n  = (y_poly_raw - y_poly_mu) / y_poly_sd

print("\nDegree 2:")
poly2 = PolynomialRegression(degree=2, learning_rate=0.1)
poly2.fit(X_poly_n, y_poly_n, epochs=2000, print_every=500)
print(f"  Weights: {np.round(poly2.weights, 4)}")
print(f"  Bias:    {poly2.bias:.4f}")
print(f"  R-squared: {poly2.r_squared(X_poly_n, y_poly_n):.4f}")

print("\nDegree 5:")
poly5 = PolynomialRegression(degree=5, learning_rate=0.1)
poly5.fit(X_poly_n, y_poly_n, epochs=2000, print_every=500)
print(f"  Weights: {np.round(poly5.weights, 4)}")
print(f"  Bias:    {poly5.bias:.4f}")
print(f"  R-squared: {poly5.r_squared(X_poly_n, y_poly_n):.4f}")

# ---- Recover original-space coefficients for degree 5 ----
print("\n--- Degree 5: Recovered original coefficients ---")
for k in range(1, 6):
    ck = poly5.weights[k-1] * y_poly_sd / (x_max ** k)
    print(f"  coef of x^{k}: {ck:+.6f}")
c0 = y_poly_mu + y_poly_sd * poly5.bias
print(f"  intercept:     {c0:+.6f}")
print("  True: c2=+0.5, c1=-2.0, c0=+3.0")

# ============================================================
# 4. RIDGE REGRESSION DEMO
# ============================================================
print("\n" + "=" * 60)
print("4. RIDGE REGRESSION (alpha=0.1)")
print("=" * 60)

ridge = RidgeRegression(n_features=3, learning_rate=0.01, alpha=0.1)
ridge.fit(X_scaled, y_scaled, epochs=1000, print_every=200)

print(f"\nRidge weights: {np.round(ridge.weights, 4)}")
print(f"Plain weights: {np.round(multi_model.weights, 4)}")
print("Ridge weights are smaller (shrunk toward zero) due to the L2 penalty.")

1. SIMPLE LINEAR REGRESSION
  Epoch    0 | Cost: 213.6543 | w: 1.2234 | b: 0.1904
  Epoch  500 | Cost: 2.1495 | w: 3.5527 | b: 1.7341
  Epoch 1000 | Cost: 2.1046 | w: 3.5055 | b: 2.0446
  Epoch 1500 | Cost: 2.1014 | w: 3.4929 | b: 2.1276

Learned: y = 3.4895x + 2.1497
True:    y = 3.5x + 2.0
R-squared: 0.9804

2. MULTIPLE LINEAR REGRESSION (3 features)
  Epoch    0 | Cost: 0.9696
  Epoch  200 | Cost: 0.1778
  Epoch  400 | Cost: 0.1773
  Epoch  600 | Cost: 0.1773
  Epoch  800 | Cost: 0.1773

Weights (standardized): [ 0.8428  0.2946 -0.2992]
Bias (standardized):    -0.0000
R-squared:              0.8227

3. POLYNOMIAL REGRESSION (degree 2 vs 5)

Degree 2:
  Epoch    0 | Cost: 1.000000
  Epoch  500 | Cost: 0.581704
  Epoch 1000 | Cost: 0.524530
  Epoch 1500 | Cost: 0.492569
  Weights: [-3.5297  5.2591]
  Bias:    -0.0071
  R-squared: 0.5253

Degree 5:
  Epoch    0 | Cost: 1.000000
  Epoch  500 | Cost: 0.436697
  Epoch 1000 | Cost: 0.433043
  Epoch 1500 | Cost: 0.432820
  Weights: [-1.6906